In [9]:
from typing import Any
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch
from skimage.measure import marching_cubes
from tqdm import tqdm

def sphericity(mask: np.ndarray, spacing=(1.0, 1.0, 1.0), min_voxels=20) -> float | None:
    """Compute the same marching-cubes sphericity used for real nodules."""
    if mask.sum() < min_voxels:
        return None

    volume = mask.sum() * np.prod(spacing)
    try:
        vertices, faces, _, _ = marching_cubes(
            mask.astype(np.float32), level=0.5, spacing=spacing
        )
    except Exception:
        return None

    if len(faces) == 0:
        return None

    triangles = vertices[faces]
    area = 0.5 * np.linalg.norm(
        np.cross(
            triangles[:, 1] - triangles[:, 0],
            triangles[:, 2] - triangles[:, 0],
        ),
        axis=1,
    ).sum()
    if area <= 0:
        return None

    value = (np.pi ** (1 / 3) * (6 * volume) ** (2 / 3)) / area
    return float(np.clip(value, 0.0, 1.0))


def surface_roughness(
    mask: np.ndarray,
    spacing=(1.0, 1.0, 1.0),
    min_voxels=20,
) -> float | None:
    """Compute the same normalized surface-area/volume roughness metric."""
    if mask.sum() < min_voxels:
        return None

    try:
        vertices, faces, _, _ = marching_cubes(
            mask.astype(np.float32), level=0.5, spacing=spacing
        )
    except Exception:
        return None

    if len(faces) == 0:
        return None

    triangles = vertices[faces]
    surface_area = 0.5 * np.linalg.norm(
        np.cross(
            triangles[:, 1] - triangles[:, 0],
            triangles[:, 2] - triangles[:, 0],
        ),
        axis=1,
    ).sum()
    volume = mask.sum() * np.prod(spacing)
    if volume <= 0 or surface_area <= 0:
        return None

    sphere_surface_area = (36 * np.pi * volume**2) ** (1 / 3)
    return float(surface_area / sphere_surface_area)


def equivalent_diameter_mm(
    volume_voxels: int,
    spacing=(1.0, 1.0, 1.0),
) -> float:
    """Compute equivalent spherical diameter from nodule volume."""
    volume_mm3 = volume_voxels * np.prod(spacing)
    return float(2 * ((3 * volume_mm3) / (4 * np.pi)) ** (1 / 3))


def normalize_conditions(condition_tuple: tuple[int, int, int]) -> torch.Tensor:
    """Convert 1-based V9 levels to a normalized (1, 3) tensor."""
    sphericity_level, spiculation_level, size_level = condition_tuple
    return torch.tensor(
        [[
            (sphericity_level - 1) / 4.0,
            (spiculation_level - 1) / 4.0,
            (size_level - 1) / 2.0,
        ]],
        dtype=torch.float32,
    )


def measure_generated_nodule(mask: np.ndarray, spacing=(1.0, 1.0, 1.0)) -> dict[str, Any]:
    """Measure one generated binary nodule mask."""
    volume_voxels = int(mask.sum())
    return {
        "nodule_voxels": volume_voxels,
        "sphericity": sphericity(mask, spacing),
        "surface_roughness": surface_roughness(mask, spacing),
        "equivalent_diameter_mm": equivalent_diameter_mm(volume_voxels, spacing),
    }

In [6]:
MASKS_DIR=Path("Z:/nas-ctm01/datasets/public/LungNodule-nifti/preprocessed/MAISI/results_synth_maskct/masks")

In [ ]:
cases_paths = sorted(str(path) for path in MASKS_DIR.glob("*.nii.gz"))

In [ ]:
mask = nib.load(cases_paths[0])
print(measure_generated_nodule(mask.get_fdata(), mask.header.get_zooms()[:3]))

{'nodule_voxels': 755, 'sphericity': 0.6662210119526963, 'surface_roughness': 1.501003393857237, 'equivalent_diameter_mm': 20.97536277770996}


In [ ]:
# Each case contains one nodule; measure all foreground voxels together.
if not cases_paths:
    raise ValueError(f"No nodule masks found in {MASKS_DIR}")

cases_measurements = []
for case_path in tqdm(cases_paths, desc="Measuring nodules"):
    case_image = nib.load(case_path)
    case_mask = case_image.get_fdata() > 0
    measurements = measure_generated_nodule(
        case_mask, spacing=case_image.header.get_zooms()[:3]
    )
    cases_measurements.append({"case_path": case_path, **measurements})

# Population standard deviation (ddof=0) across valid measurements.
print(f"Measured {len(cases_measurements)} cases; mean ± std:")
for metric in (
    "nodule_voxels",
    "sphericity",
    "surface_roughness",
    "equivalent_diameter_mm",
):
    values = np.array([
        row[metric] for row in cases_measurements
        if row[metric] is not None and np.isfinite(row[metric])
    ], dtype=float)
    if values.size:
        print(
            f"{metric}: {values.mean():.4f} ± {values.std(ddof=0):.4f} "
            f"(n={values.size}/{len(cases_measurements)})"
        )
    else:
        print(f"{metric}: unavailable (n=0/{len(cases_measurements)})")